# Влияние нефтяного рынка на курс рубля

Авторы: Мережкин Олег, Мамедов Сабухи, Заренков Пётр

## 1. Введение

В проекте сравниваем Brent, WTI и отдельный месячный блок Dubai. Цель — понять, какие нефтяные признаки лучше объясняют USD/RUB.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src.py").exists() and (ROOT.parent / "src.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display
import src

if not Path("data/model_results.csv").exists():
    src.main()

## 2. Данные

Основная дневная выборка хранится в `data/daily_features.csv`. Месячные бенчмарки с Dubai — в `data/monthly_benchmarks.csv`.

In [ ]:
daily = pd.read_csv('data/daily_features.csv', parse_dates=['date'])
monthly = pd.read_csv('data/monthly_benchmarks.csv', parse_dates=['date'])
results = pd.read_csv('data/model_results.csv')
importance = pd.read_csv('data/feature_importance.csv')

display(daily.head())
print(daily.shape)
display(monthly.head())

## 3. EDA

Сначала смотрим динамику нефти и USD/RUB. На графиках должны быть заметны кризисные периоды и резкие движения курса.

In [ ]:
for name in ['oil_prices_daily.png', 'usdrub_daily.png', 'returns_daily.png']:
    display(Image(filename=str(Path('figures') / name)))

## 4. Корреляции

Корреляционная матрица показывает общую связь между уровнями и доходностями. Важно помнить, что корреляция не доказывает причинность.

In [ ]:
display(Image(filename='figures/correlation_heatmap_daily.png'))

## 5. Лаговый анализ

Лаги от -30 до +30 дней помогают посмотреть, есть ли задержанная реакция курса на изменение нефтяных цен.

In [ ]:
display(Image(filename='figures/cross_correlation_brent.png'))
display(Image(filename='figures/cross_correlation_wti.png'))

## 6. Месячные бенчмарки

Dubai доступен как месячная серия FRED/IMF, поэтому он рассматривается отдельно от дневных моделей.

In [ ]:
display(Image(filename='figures/monthly_benchmarks.png'))

## 7. Сравнение моделей

Для всех моделей считаются MAE, RMSE и R². Train/test делятся по времени, без перемешивания.

In [ ]:
single = pd.read_csv('data/single_factor_results.csv')
cv = pd.read_csv('data/time_series_cv_results.csv')

display(results)
display(single)
display(cv.groupby('model')[['MAE', 'RMSE', 'R2']].mean().sort_values('RMSE'))
display(Image(filename='figures/model_comparison.png'))

## 8. Важность признаков

Random Forest показывает, какие нефтяные признаки чаще используются для объяснения курса.

In [ ]:
display(importance.head(20))
display(Image(filename='figures/feature_importance_rf.png'))

## 9. Прогноз на test-периоде

На графике сравнивается фактический USD/RUB и прогноз лучшей модели по RMSE.

In [ ]:
display(Image(filename='figures/forecast_actual_vs_predicted.png'))

## 10. Выводы

Нефть влияет на рубль, но не объясняет курс полностью. Brent и WTI близки между собой, поэтому возникает мультиколлинеарность. После 2014 и 2022 годов связь курса и нефти становится менее стабильной из-за санкций, ограничений и изменения структуры валютного рынка.